## Build

In [ ]:
!apt update
!apt install -y cmake build-essential git


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,696 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,394 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/unive

In [ ]:
!git clone https://github.com/marian-nmt/marian
%cd marian


Cloning into 'marian'...
remote: Enumerating objects: 60430, done.
remote: Counting objects: 100% (2351/2351), done.
remote: Compressing objects: 100% (837/837), done.
remote: Total 60430 (delta 1621), reused 2015 (delta 1457), pack-reused 58079 (from 1)
Receiving objects: 100% (60430/60430), 40.60 MiB | 12.80 MiB/s, done.
Resolving deltas: 100% (46422/46422), done.
/content/marian


In [ ]:
!mkdir build
%cd build
!cmake .. -DCOMPILE_CPU=ON -DCOMPILE_CUDA=ON
!make -j$(nproc)

/content/marian/build
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Project name: marian
-- Project version: v1.12.0+65bf82ff
Submodule 'examples' (https://github.com/m

In [1]:
#Google Driveマウントする
from google.colab import drive
drive.mount('/content/drive')

%mkdir -p /content/drive/MyDrive/marian_artifacts/bin



Mounted at /content/drive


In [ ]:
!/content/marian/build/marian --version
!/content/marian/build/marian --help | head


v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
Marian: Fast Neural Machine Translation in C++
Usage: /content/marian/build/marian [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier


In [ ]:
!cp /content/marian/build/marian* /content/drive/MyDrive/marian_artifacts/bin/
!cp /content/marian/build/spm_* /content/drive/MyDrive/marian_artifacts/bin/

In [ ]:
!ls /content/drive/MyDrive/marian_artifacts/bin


marian	     marian-decoder  marian-vocab  spm_encode	     spm_normalize
marian-conv  marian-scorer   spm_decode    spm_export_vocab  spm_train


In [2]:
#BuildしたモデルをGoogleDriveに保存しているのでそこから取ってくる
!chmod +x /content/drive/MyDrive/marian_artifacts/bin/*

In [ ]:
!ls -l /content/drive/MyDrive/marian_artifacts/bin/marian-vocab


-rwx------ 1 root root 113480200 May 29 00:46 /content/drive/MyDrive/marian_artifacts/bin/marian-vocab


##spaceあり SPM


In [ ]:
!mkdir -p /content/drive/MyDrive/assignment2
!mkdir -p /content/drive/MyDrive/assignment2/vocabs
!mkdir -p /content/drive/MyDrive/assignment2/models/6layer

In [9]:
#vocaを作る

!mkdir -p /content/vocabs


!cat /content/train14k.en \
 | /content/drive/MyDrive/marian_artifacts/bin/marian-vocab -m 14000 \
 > /content/vocabs/vocab.en14k.yml
!cp -v /content/vocabs/vocab.en14k.yml \
      /content/drive/MyDrive/assignment2/vocabs/

!cat /content/train14k.ku \
 | /content/drive/MyDrive/marian_artifacts/bin/marian-vocab -m 14000 \
 > /content/vocabs/vocab.ku14k.yml
!cp -v /content/vocabs/vocab.ku14k.yml \
      /content/drive/MyDrive/assignment2/vocabs/


[2026-06-02 03:20:47] Creating vocabulary...
[2026-06-02 03:20:47] [data] Creating vocabulary stdout from stdin
[2026-06-02 03:20:47] Finished
'/content/vocabs/vocab.en14k.yml' -> '/content/drive/MyDrive/assignment2/vocabs/vocab.en14k.yml'
[2026-06-02 03:20:47] Creating vocabulary...
[2026-06-02 03:20:47] [data] Creating vocabulary stdout from stdin
[2026-06-02 03:20:47] Finished
'/content/vocabs/vocab.ku14k.yml' -> '/content/drive/MyDrive/assignment2/vocabs/vocab.ku14k.yml'


In [ ]:
ku_voca_size = 1
en_voca_size = 1
layer_count = 5

In [ ]:
!mkdir -p /content/model

In [ ]:
import subprocess
def wc_l(path):
    return int(subprocess.check_output(["wc", "-l", str(path)]).decode().split()[0])

In [ ]:
ku_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml")
en_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml")
print(ku_voca_dim, en_voca_dim)

999 999


In [ ]:
# /content/model/ ディレクトリ内を空にする
!rm -rf /content/model/*

In [ ]:
# 訓練する
# Variables ku_voca_size, en_voca_size, ku_voca_dim, en_voca_dim must be defined
!/content/drive/MyDrive/marian_artifacts/bin/marian \
  --model /content/model/ku-en_spbpe.npz \
  --type transformer \
  --train-sets /content/train{ku_voca_size}k.ku /content/train{en_voca_size}k.en \
  --valid-sets /content/valid{ku_voca_size}k.ku /content/valid{en_voca_size}k.en \
  --vocabs /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
  --dim-vocabs {ku_voca_dim} {en_voca_dim} \
  --enc-depth {layer_count} --dec-depth {layer_count} \
  --dim-emb 512 \
  --transformer-dim-ffn 1024 \
  --transformer-heads 2 \
  --transformer-dropout 0.3 \
  --label-smoothing 0.5 \
  --max-length 200 \
  --mini-batch-fit -w 6000 \
  --optimizer adam --learn-rate 0.0003 \
  --lr-warmup 2000 --lr-decay-inv-sqrt 2000 \
  --clip-norm 1.0 \
  --valid-freq 2000 \
  --save-freq 2000 \
  --early-stopping 5 \
  --seed 42 \
  --log /content/model/train.log

[2026-05-29 01:32:07] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-05-29 01:32:07] [marian] Running on a69db31d0084 as process 36534 with command line:
[2026-05-29 01:32:07] [marian] /content/drive/MyDrive/marian_artifacts/bin/marian --model /content/model/ku-en_spbpe.npz --type transformer --train-sets /content/train1991k.ku /content/train1920k.en --valid-sets /content/valid1991k.ku /content/valid1920k.en --vocabs /content/drive/MyDrive/assignment2/vocabs/vocab.ku2k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en2k.yml --dim-vocabs 1991 1920 --enc-depth 6 --dec-depth 6 --dim-emb 256 --transformer-dim-ffn 1024 --transformer-heads 4 --transformer-dropout 0.1 --label-smoothing 0.1 --max-length 200 --mini-batch-fit -w 6000 --optimizer adam --learn-rate 0.0003 --lr-warmup 2000 --lr-decay-inv-sqrt 2000 --clip-norm 1.0 --valid-freq 2000 --save-freq 2000 --early-stopping 5 --seed 42 --log /content/model/train.log
[2026-05-29 01:32:07] [config] after: 0e
[2026-05-29 

In [ ]:
#モデルの保存
#保存先のフォルダーのvoca size
!mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k
!mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k


!cp -v /content/model/ku-en_spbpe.npz \
      /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

!cp -v /content/model/ku-en_spbpe.npz.optimizer.npz \
      /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

!cp -v /content/model/train.log \
      /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

#decorder
#vocabファイルのvaca size
#突っ込むアイヌ語のvoca size
#decorderで生成したファイルのvoca size
#保存先のフォルダーのvoca size
!/content/drive/MyDrive/marian_artifacts/bin/marian-decoder \
  -m /content/model/ku-en_spbpe.npz \
  -v /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
  --beam-size 5 --normalize 1.0 \
  < /content/test{ku_voca_size}k.ku \
  > /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en\
  2> /content/decode.err

!wc -l /content/test{ku_voca_size}k.ku /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/test{en_voca_size}k.en
!head -n 3 /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en

!cp -v /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

cp: cannot stat '/content/model/ku-en_spbpe.npz': No such file or directory
cp: cannot stat '/content/model/ku-en_spbpe.npz.optimizer.npz': No such file or directory
'/content/model/train.log' -> '/content/drive/MyDrive/assignment2/models/6layer/ku2k/en2k/train.log'
/bin/bash: line 1: 37713 Aborted                 (core dumped) /content/drive/MyDrive/marian_artifacts/bin/marian-decoder -m /content/model/ku-en_spbpe.npz -v /content/drive/MyDrive/assignment2/vocabs/vocab.ku2k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en2k.yml --beam-size 5 --normalize 1.0 < /content/test2k.ku > /content/test2k-2k.hyp.en 2> /content/decode.err
  500 /content/test2k.ku
    0 /content/test2k-2k.hyp.en
  500 /content/test2k.en
 1000 total
'/content/test2k-2k.hyp.en' -> '/content/drive/MyDrive/assignment2/models/6layer/ku2k/en2k/test2k-2k.hyp.en'


## pipeline


In [3]:
layer_count = 5

In [ ]:
!mkdir -p /content/model

import subprocess

def wc_l(path):
    return int(subprocess.check_output(["wc", "-l", str(path)]).decode().split()[0])

voca_sizes = [1, 2, 4, 6]

for ku_voca_size in voca_sizes:
    for en_voca_size in voca_sizes:

        print("=" * 80)
        print(f"START: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

        ku_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml")
        en_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml")

        print(f"ku_voca_dim = {ku_voca_dim}")
        print(f"en_voca_dim = {en_voca_dim}")

        # /content/model/ ディレクトリ内を空にする
        !rm -rf /content/model/*

        !/content/drive/MyDrive/marian_artifacts/bin/marian \
          --model /content/model/ku-en_spbpe.npz \
          --type transformer \
          --train-sets /content/train{ku_voca_size}k.ku /content/train{en_voca_size}k.en \
          --valid-sets /content/valid{ku_voca_size}k.ku /content/valid{en_voca_size}k.en \
          --vocabs /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --dim-vocabs {ku_voca_dim} {en_voca_dim} \
          --enc-depth {layer_count} --dec-depth {layer_count} \
          --dim-emb 512 \
          --transformer-dim-ffn 1024 \
          --transformer-heads 2 \
          --transformer-dropout 0.3 \
          --label-smoothing 0.5 \
          --max-length 200 \
          --mini-batch-fit -w 6000 \
          --optimizer adam --learn-rate 0.0003 \
          --lr-warmup 2000 --lr-decay-inv-sqrt 2000 \
          --clip-norm 1.0 \
          --valid-freq 2000 \
          --save-freq 2000 \
          --early-stopping 5 \
          --seed 42 \
          --log /content/model/train.log

        # モデルの保存
        # 保存先のフォルダーのvoca size
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k

        !cp -v /content/model/ku-en_spbpe.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/ku-en_spbpe.npz.optimizer.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/train.log \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        # decorder
        # vocabファイルのvaca size
        # 突っ込むアイヌ語のvoca size
        # decorderで生成したファイルのvoca size
        # 保存先のフォルダーのvoca size
        !/content/drive/MyDrive/marian_artifacts/bin/marian-decoder \
          -m /content/model/ku-en_spbpe.npz \
          -v /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --beam-size 5 --normalize 1.0 \
          < /content/test{ku_voca_size}k.ku \
          > /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en \
          2> /content/decode.err

        !wc -l /content/test{ku_voca_size}k.ku /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/test{en_voca_size}k.en
        !head -n 3 /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en

        !cp -v /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        print("=" * 80)
        print(f"DONE: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

Streaming output truncated to the last 5000 lines.
[2026-05-31 01:48:15] [data] Shuffling data
[2026-05-31 01:48:15] [data] Done reading 8,538 sentences
[2026-05-31 01:48:15] [data] Done shuffling 8,538 sentences to temp files
[2026-05-31 01:48:17] Seen 8,538 samples
[2026-05-31 01:48:17] Starting data epoch 517 in logical epoch 517
[2026-05-31 01:48:17] [data] Shuffling data
[2026-05-31 01:48:17] [data] Done reading 8,538 sentences
[2026-05-31 01:48:17] [data] Done shuffling 8,538 sentences to temp files
[2026-05-31 01:48:17] Ep. 517 : Up. 11000 : Sen. 1,831 : Cost 4.25946331 * 5,694,823 @ 7,199 after 62,802,751 : Time 62.84s : 90619.63 words/s : gNorm 0.2205
[2026-05-31 01:48:18] Seen 8,538 samples
[2026-05-31 01:48:18] Starting data epoch 518 in logical epoch 518
[2026-05-31 01:48:18] [data] Shuffling data
[2026-05-31 01:48:18] [data] Done reading 8,538 sentences
[2026-05-31 01:48:18] [data] Done shuffling 8,538 sentences to temp files
[2026-05-31 01:48:19] Seen 8,538 samples
[2026-

In [4]:
!mkdir -p /content/model

import subprocess

def wc_l(path):
    return int(subprocess.check_output(["wc", "-l", str(path)]).decode().split()[0])

en_voca_sizes = [1, 2, 4, 6, 8, 10]

for en_voca_size in en_voca_sizes:
        ku_voca_size = 8
        print("=" * 80)
        print(f"START: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

        ku_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml")
        en_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml")

        print(f"ku_voca_dim = {ku_voca_dim}")
        print(f"en_voca_dim = {en_voca_dim}")

        # /content/model/ ディレクトリ内を空にする
        !rm -rf /content/model/*

        !/content/drive/MyDrive/marian_artifacts/bin/marian \
          --model /content/model/ku-en_spbpe.npz \
          --type transformer \
          --train-sets /content/train{ku_voca_size}k.ku /content/train{en_voca_size}k.en \
          --valid-sets /content/valid{ku_voca_size}k.ku /content/valid{en_voca_size}k.en \
          --vocabs /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --dim-vocabs {ku_voca_dim} {en_voca_dim} \
          --enc-depth {layer_count} --dec-depth {layer_count} \
          --dim-emb 512 \
          --transformer-dim-ffn 1024 \
          --transformer-heads 2 \
          --transformer-dropout 0.3 \
          --label-smoothing 0.5 \
          --max-length 200 \
          --mini-batch-fit -w 6000 \
          --optimizer adam --learn-rate 0.0003 \
          --lr-warmup 2000 --lr-decay-inv-sqrt 2000 \
          --clip-norm 1.0 \
          --valid-freq 2000 \
          --save-freq 2000 \
          --early-stopping 5 \
          --seed 42 \
          --log /content/model/train.log

        # モデルの保存
        # 保存先のフォルダーのvoca size
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k

        !cp -v /content/model/ku-en_spbpe.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/ku-en_spbpe.npz.optimizer.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/train.log \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        # decorder
        # vocabファイルのvaca size
        # 突っ込むアイヌ語のvoca size
        # decorderで生成したファイルのvoca size
        # 保存先のフォルダーのvoca size
        !/content/drive/MyDrive/marian_artifacts/bin/marian-decoder \
          -m /content/model/ku-en_spbpe.npz \
          -v /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --beam-size 5 --normalize 1.0 \
          < /content/test{ku_voca_size}k.ku \
          > /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en \
          2> /content/decode.err

        !wc -l /content/test{ku_voca_size}k.ku /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/test{en_voca_size}k.en
        !head -n 3 /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en

        !cp -v /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        print("=" * 80)
        print(f"DONE: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

ストリーミング出力は最後の 5000 行に切り捨てられました。
[2026-06-02 08:20:31] [data] Shuffling data
[2026-06-02 08:20:31] [data] Done reading 8,538 sentences
[2026-06-02 08:20:31] [data] Done shuffling 8,538 sentences to temp files
[2026-06-02 08:20:32] Seen 8,538 samples
[2026-06-02 08:20:32] Starting data epoch 224 in logical epoch 224
[2026-06-02 08:20:32] [data] Shuffling data
[2026-06-02 08:20:32] [data] Done reading 8,538 sentences
[2026-06-02 08:20:32] [data] Done shuffling 8,538 sentences to temp files
[2026-06-02 08:20:33] Seen 8,538 samples
[2026-06-02 08:20:33] Starting data epoch 225 in logical epoch 225
[2026-06-02 08:20:33] [data] Shuffling data
[2026-06-02 08:20:33] [data] Done reading 8,538 sentences
[2026-06-02 08:20:33] [data] Done shuffling 8,538 sentences to temp files
[2026-06-02 08:20:34] Seen 8,538 samples
[2026-06-02 08:20:34] Starting data epoch 226 in logical epoch 226
[2026-06-02 08:20:34] [data] Shuffling data
[2026-06-02 08:20:34] [data] Done reading 8,538 sentences
[2026-06-02 08

In [11]:
!echo ku_voca_dim={ku_voca_dim}
!echo en_voca_dim={en_voca_dim}
!echo ku_voca_size={ku_voca_size}
!echo en_voca_size={en_voca_size}

ku_voca_dim=8257
en_voca_dim=5555
ku_voca_size=14
en_voca_size=14


In [5]:
!mkdir -p /content/model

import subprocess

def wc_l(path):
    return int(subprocess.check_output(["wc", "-l", str(path)]).decode().split()[0])

voca_sizes = [1, 2, 4, 6, 8, 10]

for ku_voca_size in voca_sizes:
    for en_voca_size in voca_sizes:
        if (ku_voca_size != 10) and (en_voca_size not in [8, 10]):
            continue

        print("=" * 80)
        print(f"START: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

        ku_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml")
        en_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml")

        print(f"ku_voca_dim = {ku_voca_dim}")
        print(f"en_voca_dim = {en_voca_dim}")

        # /content/model/ ディレクトリ内を空にする
        !rm -rf /content/model/*

        !/content/drive/MyDrive/marian_artifacts/bin/marian \
          --model /content/model/ku-en_spbpe.npz \
          --type transformer \
          --train-sets /content/train{ku_voca_size}k.ku /content/train{en_voca_size}k.en \
          --valid-sets /content/valid{ku_voca_size}k.ku /content/valid{en_voca_size}k.en \
          --vocabs /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --dim-vocabs {ku_voca_dim} {en_voca_dim} \
          --enc-depth {layer_count} --dec-depth {layer_count} \
          --dim-emb 512 \
          --transformer-dim-ffn 1024 \
          --transformer-heads 2 \
          --transformer-dropout 0.3 \
          --label-smoothing 0.5 \
          --max-length 200 \
          --mini-batch-fit -w 6000 \
          --optimizer adam --learn-rate 0.0003 \
          --lr-warmup 2000 --lr-decay-inv-sqrt 2000 \
          --clip-norm 1.0 \
          --valid-freq 2000 \
          --save-freq 2000 \
          --early-stopping 5 \
          --seed 42 \
          --log /content/model/train.log

        # モデルの保存
        # 保存先のフォルダーのvoca size
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k

        !cp -v /content/model/ku-en_spbpe.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/ku-en_spbpe.npz.optimizer.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/train.log \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        # decorder
        # vocabファイルのvaca size
        # 突っ込むアイヌ語のvoca size
        # decorderで生成したファイルのvoca size
        # 保存先のフォルダーのvoca size
        !/content/drive/MyDrive/marian_artifacts/bin/marian-decoder \
          -m /content/model/ku-en_spbpe.npz \
          -v /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --beam-size 5 --normalize 1.0 \
          < /content/test{ku_voca_size}k.ku \
          > /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en \
          2> /content/decode.err

        !wc -l /content/test{ku_voca_size}k.ku /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/test{en_voca_size}k.en
        !head -n 3 /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en

        !cp -v /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        print("=" * 80)
        print(f"DONE: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

ストリーミング出力は最後の 5000 行に切り捨てられました。
[2026-06-03 05:27:38] Starting data epoch 278 in logical epoch 278
[2026-06-03 05:27:38] [data] Shuffling data
[2026-06-03 05:27:38] [data] Done reading 8,538 sentences
[2026-06-03 05:27:38] [data] Done shuffling 8,538 sentences to temp files
[2026-06-03 05:27:39] Seen 8,538 samples
[2026-06-03 05:27:39] Starting data epoch 279 in logical epoch 279
[2026-06-03 05:27:39] [data] Shuffling data
[2026-06-03 05:27:39] [data] Done reading 8,538 sentences
[2026-06-03 05:27:39] [data] Done shuffling 8,538 sentences to temp files
[2026-06-03 05:27:40] Seen 8,538 samples
[2026-06-03 05:27:40] Starting data epoch 280 in logical epoch 280
[2026-06-03 05:27:40] [data] Shuffling data
[2026-06-03 05:27:40] [data] Done reading 8,538 sentences
[2026-06-03 05:27:40] [data] Done shuffling 8,538 sentences to temp files
[2026-06-03 05:27:41] Seen 8,538 samples
[2026-06-03 05:27:41] Starting data epoch 281 in logical epoch 281
[2026-06-03 05:27:41] [data] Shuffling data
[2026

In [3]:
!mkdir -p /content/model

import subprocess

def wc_l(path):
    return int(subprocess.check_output(["wc", "-l", str(path)]).decode().split()[0])



for layer_count in [3, 4, 6, 7, 8]:
        ku_voca_size = 6
        en_voca_size = 6
        print("=" * 80)
        print(f"START: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

        ku_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml")
        en_voca_dim = wc_l(f"/content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml")

        print(f"ku_voca_dim = {ku_voca_dim}")
        print(f"en_voca_dim = {en_voca_dim}")

        # /content/model/ ディレクトリ内を空にする
        !rm -rf /content/model/*

        !/content/drive/MyDrive/marian_artifacts/bin/marian \
          --model /content/model/ku-en_spbpe.npz \
          --type transformer \
          --train-sets /content/train{ku_voca_size}k.ku /content/train{en_voca_size}k.en \
          --valid-sets /content/valid{ku_voca_size}k.ku /content/valid{en_voca_size}k.en \
          --vocabs /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --dim-vocabs {ku_voca_dim} {en_voca_dim} \
          --enc-depth {layer_count} --dec-depth {layer_count} \
          --dim-emb 512 \
          --transformer-dim-ffn 1024 \
          --transformer-heads 2 \
          --transformer-dropout 0.3 \
          --label-smoothing 0.5 \
          --max-length 200 \
          --mini-batch-fit -w 6000 \
          --optimizer adam --learn-rate 0.0003 \
          --lr-warmup 2000 --lr-decay-inv-sqrt 2000 \
          --clip-norm 1.0 \
          --valid-freq 2000 \
          --save-freq 2000 \
          --early-stopping 5 \
          --seed 42 \
          --log /content/model/train.log

        # モデルの保存
        # 保存先のフォルダーのvoca size
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k
        !mkdir -p /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k

        !cp -v /content/model/ku-en_spbpe.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/ku-en_spbpe.npz.optimizer.npz \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        !cp -v /content/model/train.log \
              /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        # decorder
        # vocabファイルのvaca size
        # 突っ込むアイヌ語のvoca size
        # decorderで生成したファイルのvoca size
        # 保存先のフォルダーのvoca size
        !/content/drive/MyDrive/marian_artifacts/bin/marian-decoder \
          -m /content/model/ku-en_spbpe.npz \
          -v /content/drive/MyDrive/assignment2/vocabs/vocab.ku{ku_voca_size}k.yml /content/drive/MyDrive/assignment2/vocabs/vocab.en{en_voca_size}k.yml \
          --beam-size 5 --normalize 1.0 \
          < /content/test{ku_voca_size}k.ku \
          > /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en \
          2> /content/decode.err

        !wc -l /content/test{ku_voca_size}k.ku /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/test{en_voca_size}k.en
        !head -n 3 /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en

        !cp -v /content/test{ku_voca_size}k-{en_voca_size}k.hyp.en /content/drive/MyDrive/assignment2/models/{layer_count}layer/ku{ku_voca_size}k/en{en_voca_size}k/

        print("=" * 80)
        print(f"DONE: ku{ku_voca_size}k -> en{en_voca_size}k")
        print("=" * 80)

ストリーミング出力は最後の 5000 行に切り捨てられました。
[2026-06-03 10:02:12] [config] transformer-aan-activation: swish
[2026-06-03 10:02:12] [config] transformer-aan-depth: 2
[2026-06-03 10:02:12] [config] transformer-aan-nogate: false
[2026-06-03 10:02:12] [config] transformer-decoder-autoreg: self-attention
[2026-06-03 10:02:12] [config] transformer-decoder-dim-ffn: 0
[2026-06-03 10:02:12] [config] transformer-decoder-ffn-depth: 0
[2026-06-03 10:02:12] [config] transformer-depth-scaling: false
[2026-06-03 10:02:12] [config] transformer-dim-aan: 2048
[2026-06-03 10:02:12] [config] transformer-dim-ffn: 1024
[2026-06-03 10:02:12] [config] transformer-dropout: 0.3
[2026-06-03 10:02:12] [config] transformer-dropout-attention: 0
[2026-06-03 10:02:12] [config] transformer-dropout-ffn: 0
[2026-06-03 10:02:12] [config] transformer-ffn-activation: swish
[2026-06-03 10:02:12] [config] transformer-ffn-depth: 2
[2026-06-03 10:02:12] [config] transformer-guided-alignment-layer: last
[2026-06-03 10:02:12] [config] trans